# 사전 준비

In [5]:
root = './data_tensor_cache'

In [6]:
import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

def get_initial_data(random_state=42):
    '''
    missing_corrected.csv를 불러오고,
    데이터 스플릿까지 하는 함수
    Args:
        random_state(int, optional): 데이터 스플릿할 때 필요한 랜덤 시드
    '''
    CURDIR = os.path.dirname(__file__)
    DATA_PATH = os.path.join(CURDIR, 'missing_corrected.csv')
    DATA = pd.read_csv(DATA_PATH)
    DATA.head()

    # 범주형 변수 더미화 시 train/test 간 불일치를 방지하기 위해
    # 스플릿 전에 전체 데이터 기준으로 가능한 범주를 정의하고 CategoricalDtype으로 고정
    for col in DATA.columns:
        cats = list(DATA[col].unique())
        DATA[col] = DATA[col].astype(pd.api.types.CategoricalDtype(categories=cats))

    # validation set, test set이 imputation 설계하는 데 들어가면 안 됨, 일종의 사후판단 정보가 들어갈 수 있기 때문
    y = DATA['REASONb']
    X = DATA.drop('REASONb', axis=1)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def get_initial_data_sampled(size=100, random_state=42):
    '''
    missing_corrected.csv를 불러오고,
    @@전체는 너무 많으니까 잘 돌아가는지 보기 위해 조금만 추출해서@@
    데이터 스플릿까지 하는 함수
    Args:
        random_state(int, optional): 데이터 스플릿할 때 필요한 랜덤 시드
    '''
    CURDIR = os.path.dirname(__file__)
    DATA_PATH = os.path.join(CURDIR, 'missing_corrected.csv')
    DATA = pd.read_csv(DATA_PATH)
    DATA.head()

    # 범주형 변수 더미화 시 train/test 간 불일치를 방지하기 위해
    # 스플릿 전에 전체 데이터 기준으로 가능한 범주를 정의하고 CategoricalDtype으로 고정
    for col in DATA.columns:
        cats = list(DATA[col].unique())
        DATA[col] = DATA[col].astype(pd.api.types.CategoricalDtype(categories=cats))

    DATA = DATA.iloc[:size]

    # validation set, test set이 imputation 설계하는 데 들어가면 안 됨, 일종의 사후판단 정보가 들어갈 수 있기 때문
    y = DATA['REASONb']
    X = DATA.drop('REASONb', axis=1)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

def get_initial_data_sampled_stratified(size=100, random_state=42):
    '''
    missing_corrected.csv를 불러오고,
    @@타겟 변수 'REASONb'를 기준으로 계층적 샘플링을 통해 지정된 크기(size)만큼 추출 후@@
    학습, 검증, 테스트 세트로 분할하는 함수
    Args:
        size(int, optional): 추출할 최종 데이터셋의 크기
        random_state(int, optional): 데이터 스플릿할 때 필요한 랜덤 시드
    '''
    CURDIR = os.path.dirname(__file__)
    DATA_PATH = os.path.join(CURDIR, 'missing_corrected.csv')
    DATA = pd.read_csv(DATA_PATH)
    DATA.head()

    # 범주형 변수 더미화 시 train/test 간 불일치를 방지하기 위해
    # 스플릿 전에 전체 데이터 기준으로 가능한 범주를 정의하고 CategoricalDtype으로 고정
    for col in DATA.columns:
        cats = list(DATA[col].unique())
        DATA[col] = DATA[col].astype(pd.api.types.CategoricalDtype(categories=cats))

    # y와 X 분리
    y_full = DATA['REASONb']
    X_full = DATA.drop('REASONb', axis=1)


    # 2. 🎯 계층적 샘플링을 통한 데이터 추출 (iloc 대체)
    # 전체 데이터셋이 size보다 작을 경우, 전체 데이터를 사용합니다.
    if len(X_full) <= size:
        print(f"Warning: Dataset size ({len(X_full)}) is smaller than requested size ({size}). Using full dataset.")
        X_sampled = X_full
        y_sampled = y_full
    else:
        # train_test_split을 활용하여 'y_full'을 기준으로 계층적 추출
        # test_size를 이용하여 size만큼의 비율을 계산
        sample_ratio = size / len(X_full)

        # NOTE: 추출할 데이터셋을 X_temp로, 버릴 데이터를 X_discard로 간주합니다.
        # test_size = 1 - sample_ratio 를 해야 sample_ratio 만큼의 데이터가 남게 됨
        X_discard, X_sampled, y_discard, y_sampled = train_test_split(
            X_full, y_full, test_size=sample_ratio, random_state=random_state, stratify=y_full
        )
        print(f"Successfully sampled {len(X_sampled)} records using stratification.")


    # 3. 추출된 데이터셋을 학습(70%), 임시(30%)로 계층 분할
    X_train, X_temp, y_train, y_temp = train_test_split(
        X_sampled, y_sampled, test_size=0.30, random_state=random_state, stratify=y_sampled
    )

    # 4. 임시 데이터셋을 검증(50%), 테스트(50%)로 계층 분할
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=random_state, stratify=y_temp
    )

    # 🚨 중요: 범주형 변수의 범주 정보가 drop되지 않도록 처리
    # 만약 범주형 변수의 범주가 추출된 데이터에 모두 없다면 오류가 날 수 있으므로
    # 이후 데이터 전처리 단계에서 CategoricalDtype을 다시 설정하는 것이 안전합니다.
    # 현재는 원본의 cats를 유지한 채 추출합니다.

    # 추출된 데이터의 클래스 분포 확인 (선택 사항)
    print("\n--- Final Split Distribution (Target: REASONb) ---")
    print(f"Train Set: {y_train.value_counts(normalize=True).to_dict()}")
    print(f"Validation Set: {y_val.value_counts(normalize=True).to_dict()}")
    print(f"Test Set: {y_test.value_counts(normalize=True).to_dict()}")

    return X_train, X_val, X_test, y_train, y_val, y_test

def fully_connected_edge_index(num_nodes, self_loops=False):
    '''
    이름 그대로, num_nodes 즉 변수 개수만 알면 됨
    '''
    nodes = torch.arange(num_nodes)
    row, col = torch.meshgrid(nodes, nodes, indexing="ij")
    edge_index = torch.stack([row.reshape(-1), col.reshape(-1)], dim=0)
    if not self_loops:
        mask = row != col
        edge_index = edge_index[:, mask.reshape(-1)]
    return edge_index

def fully_connected_edge_index_batched(num_nodes, batch_size, self_loops=False):
    '''
    batched edge_index는 결국 옆으로 이어붙인 것일 뿐, 즉 shape: [2, sum(num_edges{i})]
    '''
    single = fully_connected_edge_index(num_nodes=num_nodes)
    batch_list = [single for i in range(batch_size)]
    return torch.concatenate(batch_list, dim=1)

def mi_edge_index(mi_dict_path, top_k=6, return_edge_attr=False):
    """
    mi_dict: {col_name: pd.Series} (index=다른 변수명, value=MI, 내림차순 정렬)
    top_k: 각 src에서 MI 상위 k개로만 유향 엣지(src->dst) 생성
    return_edge_attr: True면 edge_attr로 MI 가중치 반환
    """

    import pickle
    with open(mi_dict_path, 'rb') as f:
        mi_dict = pickle.load(f)

    cols = list(mi_dict.keys())
    col_to_idx = {c: i for i, c in enumerate(cols)}

    src_idx, dst_idx, weights = [], [], []

    for src in cols:
        series = mi_dict[src]

        # 우리 그래프에 존재하는 변수만 남기고, 자기 자신은 제외
        series = series[series.index.isin(cols)]
        if src in series.index:
            series = series.drop(index=src)

        # 상위 k개 선택 (이미 내림차순 정렬되어 있다고 가정)
        top_neighbors = series.head(top_k)

        for dst, w in top_neighbors.items():
            src_idx.append(col_to_idx[src])
            dst_idx.append(col_to_idx[dst])
            if return_edge_attr:
                weights.append(float(w))

    edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)

    if return_edge_attr:
        edge_attr = torch.tensor(weights, dtype=torch.float)
        return edge_index, edge_attr

    return edge_index

def mi_edge_index_batched(batch_size, mi_dict_path, top_k=6, return_edge_attr=False)->torch.Tensor:
    single = mi_edge_index(mi_dict_path=mi_dict_path, top_k=top_k, return_edge_attr=return_edge_attr)

    batch_list = [single for _ in range(batch_size)]

    return torch.concatenate(tensors=batch_list, dim=1)

def get_col_dims(df: pd.DataFrame):
    '''
    변수별 범주의 개수 파악
    '''
    col_dims = [len(df[col].unique()) for col in df.columns]
    return col_dims

def get_ad_dis_col(df:pd.DataFrame):
    '''
    admission 시의 컬럼, discharge 시의 컬럼을 나누어 리턴
    Args:
        df(pd.DataFrame): 원본 데이터프레임, REASONb는 자동으로 제외됨
    Returns:
        (admission 시의 컬럼 list, discharge 시의 컬럼 list)
    '''
    cols = list(df.columns)

    if 'LOS' in cols:
        cols.remove('LOS')

    if 'REASONb' in cols:
        cols.remove('REASONb')

    change = []
    change_D = []

    for i in cols:
        if i.endswith('_D'):
            change_D.append(i)
            change.append(i[:-2])

    ad = [i for i in cols if i not in change_D]
    dis = ad.copy()
    for i in range(len(ad)):
        if dis[i] in change:
            dis[i] = dis[i] + '_D'

    return ad, dis

def find_indices(lst, targets):
    return [lst.index(t) if t in lst else None for t in targets]

def get_ad_dis_index(df: pd.DataFrame):
    col_list = list(df.columns)
    ad, dis = get_ad_dis_col(df)
    ad_col_index = find_indices(col_list, ad)
    dis_col_index = find_indices(col_list, dis)
    return ad_col_index, dis_col_index

def get_col_info(df: pd.DataFrame):
    '''
    Returns: (tuple)
        col_list, col_dims, ad_col_index, dis_col_index

        col_list: 보관용, 데이터에 등장하는 열 이름의 순서
        col_dims: col_list 순서대로 변수별 범주의 개수
        ad_col_index: admission에 해당하는 변수의 integer position
        dis_col_index: discharge에 해당하는 변수의 integer position
    '''
    col_list = list(df.columns)
    col_dims = get_col_dims(df)
    ad_col_index, dis_col_index = get_ad_dis_index(df)
    return col_list, col_dims, ad_col_index, dis_col_index

def organize_labels(df: pd.DataFrame):
    '''
    -9가 있는 변수를 그대로 엔티티 임베딩에 넣으면 이상해짐
    왜냐하면 엔티티 임베딩 모델은 레이블들이 연속된 정수들의 범위로 있다고 가정하기 때문
    -9, 1, 2, 3 이렇게 있었다면
    -9, -8, -7, -6, -5, ~~~ 이런 것으로 가정함

    -9, 1, 2, 3를
    0, 1, 2, 3으로 바꿈 (-9 -> 4)

    + CBSA2020
    이것도 문제가 됨
    10000 24242 32646 75577 이런 식이라 연속된 정수들의 레이블이 아님
    10000 24242 32646 75577 -> 1, 2, 3, 4
    '''

    for col in df.columns:
        labels = sorted(df[col].unique())
        replace_dict = {labels[i]: i for i in range(len(labels))}
        df[col] = df[col].replace(replace_dict)

    return df

def df_to_tensor(df: pd.DataFrame | pd.Series, dtype=torch.long):
    df_np = df.to_numpy()
    return torch.tensor(df_np, dtype=dtype)

def device_set():
    device = torch.device('cpu')

    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.mps.is_available():
        os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
        device = torch.device('mps')

    print(f'Using device: {device}')

    return device


In [7]:
'''
결국 pyg Data / Batch를 생성해서 저장 또는 로드할 필요가 없었고, 오히려 불편함, 저장공간 낭비였음
37개의 타임스탬프를 미리 만들어 저장하는 것은 원본 데이터의 37배에 달하는 용량으로 뻥튀기가 되는 것
우리는 어차피 필요한 게 1. 입소 시, 2. 퇴소 시, 3. 제로 패딩 이기 때문에
1,2만 들고 엔티티 임베딩까지 한 뒤에
모델 입력하기 직전에 시계열 데이터로 변환하면 됨

이 모든 걸 텐서로만 진행할 수 있음 + pyg Data / Batch를 쓰면 기본 알고리즘 때문에 쉐입이 이상해지던가 객체를 생성해서 저장하기 때문에 용량이 더 커짐


독스트링은 LLM에 의해 생성됨
'''
import os
import torch
import pandas as pd

from torch.utils.data import Dataset, DataLoader

class TEDSTensorDataset(Dataset):
    """
    TEDS (Temporal Embedding Deep Sequence) 모델을 위한 PyTorch Dataset.

    원본 데이터를 pyg.Data/Batch 대신 **순수 PyTorch Tensor** 형태로 변환하고 저장하여,
    저장 공간 낭비와 불필요한 객체 생성 오버헤드를 방지합니다. 데이터에는 입소(Admission)
    시점과 퇴소(Discharge) 시점 데이터만이 포함됩니다.

    Attributes:
        root (str): 데이터 저장 및 로드 경로의 루트 디렉토리.
        processed_tensor (torch.Tensor): 전처리된 최종 데이터 텐서 (입력 데이터 + 레이블).
                                         Shape: (num_samples, num_features).
                                         DataLoader에게 이걸 X, y로 나누어 전달하게 됨
        col_info (tuple[list[int], list[int]]): 컬럼 정보.
                                                (입소 시점 컬럼 인덱스 리스트, 퇴소 시점 컬럼 인덱스 리스트)
        LOS (pandas.Series): Length of Stay (재원 일수) 정보.
    """
    def __init__(self, root):
        """
        TEDSTensorDataset의 생성자.

        데이터 경로 설정, 디렉토리 생성 후, 전처리된 데이터를 로드하거나
        새로운 전처리 과정을 수행하여 데이터를 메모리에 로드합니다.

        Args:
            root: 데이터가 위치할 루트 디렉토리 경로.
        """
        super().__init__()
        self.root = root

        self.raw_dir = os.path.join(root, "raw")
        if not os.path.exists(self.raw_dir):
            os.mkdir(self.raw_dir)

        self.process_dir = os.path.join(root, 'process')
        if not os.path.exists(self.process_dir):
            os.mkdir(self.process_dir)

        processed_data_path = os.path.join(self.process_dir, "processed_data.pt")
        if os.path.exists(processed_data_path):
            print("저장되어 있는 전처리된 데이터가 있습니다. 해당 데이터를 불러오는 중..")
            self.processed_tensor, self.col_info, self.LOS = torch.load(processed_data_path, weights_only=False)
            print("불러오기 완료")
        else:
            print("저장되어 있는 전처리된 데이터가 없으므로, 전처리 과정을 진행합니다. 전처리된 데이터는 저장됩니다.")
            print(f"저장 경로: {processed_data_path}")
            processed_data = self.process
            print("전처리 완료")
            self.processed_tensor, self.col_info, self.LOS = processed_data
            print("불러오기 완료")
            torch.save(processed_data, processed_data_path)
            print("전처리된 데이터 저장 완료")

    def __getitem__(self, index):
        """
        주어진 인덱스에 해당하는 하나의 샘플과 레이블을 반환합니다.

        Args:
            index: 데이터 셋 내의 샘플 인덱스.

        Returns:
            (input_tensor, y_label): 입력 텐서와 레이블 텐서의 튜플.
        """
        input_tensor = self.processed_tensor[index, :-1]
        y_label = self.processed_tensor[index, -1]
        los = self.LOS[index]
        return input_tensor, y_label, los

    def __len__(self):
        """
        데이터 셋의 전체 샘플 개수를 반환합니다.

        Returns:
            데이터 셋의 크기 (샘플 개수).
        """
        return self.processed_tensor.shape[0]

    @property
    def process(self):
        """
        원본 데이터를 읽고 전처리 과정을 수행합니다.

        전처리 단계:
        1. CSV 파일 로드 ('missing_corrected.csv')
        2. 'LOS' (Length of Stay) 컬럼 분리 및 제거
        3. 레이블 ('REASONb') 정리 (organize_labels)
        4. 컬럼 정보 추출 (get_col_info)
        5. Pandas DataFrame을 PyTorch Tensor로 변환 (df_to_tensor)

        Raises:
            ValueError: 원본 데이터에 'LOS' 또는 'REASONb' 컬럼이 없을 경우 발생.

        Returns:
            (df_tensor, col_info, LOS): 전처리된 데이터 튜플.
        """
        data_path = os.path.join(self.raw_dir, 'missing_corrected.csv')
        df = pd.read_csv(data_path)

        # los 따로 빼기
        if 'LOS' in df.columns:
            LOS = df['LOS']
            LOS = df_to_tensor(LOS)
            df = df.drop('LOS', axis=1)
        else:
            raise ValueError('raw data에서 LOS 데이터를 찾을 수 없습니다.')

        if 'REASONb' not in df.columns:
            raise ValueError('raw data에서 REASONb 데이터를 찾을 수 없습니다.')

        # label_organize
        df = organize_labels(df)
        # df to tensor
        df_tensor = df_to_tensor(df)

        # get col infos, list of (col_list, col_dims, ad_col_index, dis_col_index)
        # ad_col_index, dis_col_index는 다음과 같음 integer position of admission col, discharge col
        df = df.drop("REASONb", axis=1)
        col_info = get_col_info(df)

        # processed_data는 (tensor, col_info, LOS)형태
        # LOS는 pd.Series임
        # col_info는 다음과 같음 (col_list, col_dims, ad_col_index, dis_col_index)
        return df_tensor, col_info, LOS # -> self.process하면 tuple로 반환될 것


In [8]:
dataset = TEDSTensorDataset(root)
dataloader = DataLoader(dataset, 32, shuffle=True, num_workers=8)
counter = 0
print("예시 출력:")
for batch in dataloader:
    if counter == 10: break
    print(batch[-1])
    counter += 1

저장되어 있는 전처리된 데이터가 있습니다. 해당 데이터를 불러오는 중..
불러오기 완료
예시 출력:


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/miniconda3/envs/pyg_env/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/miniconda3/envs/pyg_env/lib/python3.10/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'TEDSTensorDataset' on <module '__main__' (built-in)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/miniconda3/envs/pyg_env/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/miniconda3/envs/pyg_env/lib/python3.10/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'TEDSTensorDataset' on <module '__main__' (built-in)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/minicon

RuntimeError: DataLoader worker (pid(s) 2363) exited unexpectedly

In [9]:
import torch
from torch_geometric.nn import GCNConv


class TGCN(torch.nn.Module):
    r"""An implementation of the Temporal Graph Convolutional Gated Recurrent Cell.
    For details see this paper: `"T-GCN: A Temporal Graph ConvolutionalNetwork for
    Traffic Prediction." <https://arxiv.org/abs/1811.05320>`_

    Args:
        in_channels (int): Number of input features.
        out_channels (int): Number of output features.
        improved (bool): Stronger self loops. Default is False.
        cached (bool): Caching the message weights. Default is False.
        add_self_loops (bool): Adding self-loops for smoothing. Default is True.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        improved: bool = False,
        cached: bool = False,
        add_self_loops: bool = True,
    ):
        super(TGCN, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.improved = improved
        self.cached = cached
        self.add_self_loops = add_self_loops

        self._create_parameters_and_layers()

    def _create_update_gate_parameters_and_layers(self):

        self.conv_z = GCNConv(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            improved=self.improved,
            cached=self.cached,
            add_self_loops=self.add_self_loops,
        )

        self.linear_z = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_reset_gate_parameters_and_layers(self):

        self.conv_r = GCNConv(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            improved=self.improved,
            cached=self.cached,
            add_self_loops=self.add_self_loops,
        )

        self.linear_r = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_candidate_state_parameters_and_layers(self):

        self.conv_h = GCNConv(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            improved=self.improved,
            cached=self.cached,
            add_self_loops=self.add_self_loops,
        )

        self.linear_h = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_parameters_and_layers(self):
        self._create_update_gate_parameters_and_layers()
        self._create_reset_gate_parameters_and_layers()
        self._create_candidate_state_parameters_and_layers()

    def _set_hidden_state(self, X, H):
        if H is None:
            H = torch.zeros(X.shape[0], self.out_channels).to(X.device)
        return H

    def _calculate_update_gate(self, X, edge_index, edge_weight, H):
        Z = torch.cat([self.conv_z(X, edge_index, edge_weight), H], axis=1)
        Z = self.linear_z(Z)
        Z = torch.sigmoid(Z)
        return Z

    def _calculate_reset_gate(self, X, edge_index, edge_weight, H):
        R = torch.cat([self.conv_r(X, edge_index, edge_weight), H], axis=1)
        R = self.linear_r(R)
        R = torch.sigmoid(R)
        return R

    def _calculate_candidate_state(self, X, edge_index, edge_weight, H, R):
        H_tilde = torch.cat([self.conv_h(X, edge_index, edge_weight), H * R], axis=1)
        H_tilde = self.linear_h(H_tilde)
        H_tilde = torch.tanh(H_tilde)
        return H_tilde

    def _calculate_hidden_state(self, Z, H, H_tilde):
        H = Z * H + (1 - Z) * H_tilde
        return H

    def forward(
        self,
        X: torch.FloatTensor,
        edge_index: torch.LongTensor,
        edge_weight: torch.FloatTensor = None,
        H: torch.FloatTensor = None,
    ) -> torch.FloatTensor:
        """
        Making a forward pass. If edge weights are not present the forward pass
        defaults to an unweighted graph. If the hidden state matrix is not present
        when the forward pass is called it is initialized with zeros.

        Arg types:
            * **X** *(PyTorch Float Tensor)* - Node features.
            * **edge_index** *(PyTorch Long Tensor)* - Graph edge indices.
            * **edge_weight** *(PyTorch Long Tensor, optional)* - Edge weight vector.
            * **H** *(PyTorch Float Tensor, optional)* - Hidden state matrix for all nodes.

        Return types:
            * **H** *(PyTorch Float Tensor)* - Hidden state matrix for all nodes.
        """
        H = self._set_hidden_state(X, H)
        Z = self._calculate_update_gate(X, edge_index, edge_weight, H)
        R = self._calculate_reset_gate(X, edge_index, edge_weight, H)
        H_tilde = self._calculate_candidate_state(X, edge_index, edge_weight, H, R)
        H = self._calculate_hidden_state(Z, H, H_tilde)
        return H


class TGCN2(torch.nn.Module):
    r"""An implementation THAT SUPPORTS BATCHES of the Temporal Graph Convolutional Gated Recurrent Cell.
    For details see this paper: `"T-GCN: A Temporal Graph ConvolutionalNetwork for
    Traffic Prediction." <https://arxiv.org/abs/1811.05320>`_

    Args:
        in_channels (int): Number of input features.
        out_channels (int): Number of output features.
        batch_size (int): Size of the batch.
        improved (bool): Stronger self loops. Default is False.
        cached (bool): Caching the message weights. Default is False.
        add_self_loops (bool): Adding self-loops for smoothing. Default is True.
    """

    def __init__(self, in_channels: int, out_channels: int,
                 batch_size: int,  # this entry is unnecessary, kept only for backward compatibility
                 improved: bool = False, cached: bool = False,
                 add_self_loops: bool = True):
        super(TGCN2, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.improved = improved
        self.cached = cached
        self.add_self_loops = add_self_loops
        self.batch_size = batch_size  # not needed
        self._create_parameters_and_layers()

    def _create_update_gate_parameters_and_layers(self):
        self.conv_z = GCNConv(in_channels=self.in_channels,  out_channels=self.out_channels, improved=self.improved,
                              cached=self.cached, add_self_loops=self.add_self_loops )
        self.linear_z = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_reset_gate_parameters_and_layers(self):
        self.conv_r = GCNConv(in_channels=self.in_channels, out_channels=self.out_channels, improved=self.improved,
                              cached=self.cached, add_self_loops=self.add_self_loops )
        self.linear_r = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_candidate_state_parameters_and_layers(self):
        self.conv_h = GCNConv(in_channels=self.in_channels, out_channels=self.out_channels, improved=self.improved,
                              cached=self.cached, add_self_loops=self.add_self_loops )
        self.linear_h = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_parameters_and_layers(self):
        self._create_update_gate_parameters_and_layers()
        self._create_reset_gate_parameters_and_layers()
        self._create_candidate_state_parameters_and_layers()

    def _set_hidden_state(self, X, H):
        if H is None:
            # can infer batch_size from X.shape, because X is [B, N, F]
            H = torch.zeros(X.shape[0], X.shape[1], self.out_channels).to(X.device) #(b, 207, 32)
        return H

    def _calculate_update_gate(self, X, edge_index, edge_weight, H):
        Z = torch.cat([self.conv_z(X, edge_index, edge_weight), H], axis=2) # (b, 207, 64)
        Z = self.linear_z(Z) # (b, 207, 32)
        Z = torch.sigmoid(Z)

        return Z

    def _calculate_reset_gate(self, X, edge_index, edge_weight, H):
        R = torch.cat([self.conv_r(X, edge_index, edge_weight), H], axis=2) # (b, 207, 64)
        R = self.linear_r(R) # (b, 207, 32)
        R = torch.sigmoid(R)

        return R

    def _calculate_candidate_state(self, X, edge_index, edge_weight, H, R):
        H_tilde = torch.cat([self.conv_h(X, edge_index, edge_weight), H * R], axis=2) # (b, 207, 64)
        H_tilde = self.linear_h(H_tilde) # (b, 207, 32)
        H_tilde = torch.tanh(H_tilde)

        return H_tilde

    def _calculate_hidden_state(self, Z, H, H_tilde):
        H = Z * H + (1 - Z) * H_tilde   # # (b, 207, 32)
        return H

    def forward(self,X: torch.FloatTensor, edge_index: torch.LongTensor, edge_weight: torch.FloatTensor = None,
                H: torch.FloatTensor = None ) -> torch.FloatTensor:
        """
        Making a forward pass. If edge weights are not present the forward pass
        defaults to an unweighted graph. If the hidden state matrix is not present
        when the forward pass is called it is initialized with zeros.

        Arg types:
            * **X** *(PyTorch Float Tensor)* - Node features.
            * **edge_index** *(PyTorch Long Tensor)* - Graph edge indices.
            * **edge_weight** *(PyTorch Long Tensor, optional)* - Edge weight vector.
            * **H** *(PyTorch Float Tensor, optional)* - Hidden state matrix for all nodes.

        Return types:
            * **H** *(PyTorch Float Tensor)* - Hidden state matrix for all nodes.
        """
        H = self._set_hidden_state(X, H)
        Z = self._calculate_update_gate(X, edge_index, edge_weight, H)
        R = self._calculate_reset_gate(X, edge_index, edge_weight, H)
        H_tilde = self._calculate_candidate_state(X, edge_index, edge_weight, H, R)
        H = self._calculate_hidden_state(Z, H, H_tilde) # (b, 207, 32)
        return H

In [10]:
import torch

import sys
import os

class A3TGCN(torch.nn.Module):
    r"""An implementation of the Attention Temporal Graph Convolutional Cell.
    For details see this paper: `"A3T-GCN: Attention Temporal Graph Convolutional
    Network for Traffic Forecasting." <https://arxiv.org/abs/2006.11583>`_

    Args:
        in_channels (int): Number of input features.
        out_channels (int): Number of output features.
        periods (int): Number of time periods.
        improved (bool): Stronger self loops (default :obj:`False`).
        cached (bool): Caching the message weights (default :obj:`False`).
        add_self_loops (bool): Adding self-loops for smoothing (default :obj:`True`).
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        periods: int,
        improved: bool = False,
        cached: bool = False,
        add_self_loops: bool = True
    ):
        super(A3TGCN, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.periods = periods
        self.improved = improved
        self.cached = cached
        self.add_self_loops = add_self_loops
        self._setup_layers()

    def _setup_layers(self):
        self._base_tgcn = TGCN(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            improved=self.improved,
            cached=self.cached,
            add_self_loops=self.add_self_loops,
        )
        device = device_set()
        self._attention = torch.nn.Parameter(torch.empty(self.periods, device=device))
        torch.nn.init.uniform_(self._attention)

    def forward(
        self,
        X: torch.FloatTensor,
        edge_index: torch.LongTensor,
        edge_weight: torch.FloatTensor | None = None,
        H: torch.FloatTensor | None = None,
    ) -> torch.FloatTensor:
        """
        Making a forward pass. If edge weights are not present the forward pass
        defaults to an unweighted graph. If the hidden state matrix is not present
        when the forward pass is called it is initialized with zeros.

        Arg types:
            * **X** (PyTorch Float Tensor): Node features for T time periods.
            * **edge_index** (PyTorch Long Tensor): Graph edge indices.
            * **edge_weight** (PyTorch Long Tensor, optional)*: Edge weight vector.
            * **H** (PyTorch Float Tensor, optional): Hidden state matrix for all nodes.

        Return types:
            * **H** (PyTorch Float Tensor): Hidden state matrix for all nodes.
        """
        H_accum = 0
        H_sequence_outputs = []

        probs = torch.nn.functional.softmax(self._attention, dim=0)

        H_previous = H

        for period in range(self.periods):
            X_current = X[:, :, :, period]
            H_current = self._base_tgcn(X_current, edge_index, edge_weight, H_previous)
            H_previous = H_current
            H_sequence_outputs.append(probs[period] * H_current)

        H_accum = sum(H_sequence_outputs)

        return H_accum



class A3TGCN2(torch.nn.Module):
    r"""An implementation THAT SUPPORTS BATCHES of the Attention Temporal Graph Convolutional Cell.
    For details see this paper: `"A3T-GCN: Attention Temporal Graph Convolutional
    Network for Traffic Forecasting." <https://arxiv.org/abs/2006.11583>`_

    Args:
        in_channels (int): Number of input features.
        out_channels (int): Number of output features.
        periods (int): Number of time periods.
        improved (bool): Stronger self loops (default :obj:`False`).
        cached (bool): Caching the message weights (default :obj:`False`).
        add_self_loops (bool): Adding self-loops for smoothing (default :obj:`True`).
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        periods: int,
        batch_size:int,
        improved: bool = False,
        cached: bool = False,
        add_self_loops: bool = True):
        super(A3TGCN2, self).__init__()

        self.in_channels = in_channels  # 2
        self.out_channels = out_channels # 32
        self.periods = periods # 12
        self.improved = improved
        self.cached = cached
        self.add_self_loops = add_self_loops
        self.batch_size = batch_size
        self._setup_layers()

    def _setup_layers(self):
        self._base_tgcn = TGCN2(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            batch_size=self.batch_size,
            improved=self.improved,
            cached=self.cached,
            add_self_loops=self.add_self_loops)

        device = device_set()
        self._attention = torch.nn.Parameter(torch.empty(self.periods, device=device))
        torch.nn.init.uniform_(self._attention)

    def forward(
        self,
        X: torch.FloatTensor,
        edge_index: torch.LongTensor,
        edge_weight: torch.FloatTensor | None = None,
        H: torch.FloatTensor | None  = None
    ) -> torch.FloatTensor:
        """
        Making a forward pass. If edge weights are not present the forward pass
        defaults to an unweighted graph. If the hidden state matrix is not present
        when the forward pass is called it is initialized with zeros.

        Arg types:
            * **X** (PyTorch Float Tensor): Node features for T time periods.
            * **edge_index** (PyTorch Long Tensor): Graph edge indices.
            * **edge_weight** (PyTorch Long Tensor, optional)*: Edge weight vector.
            * **H** (PyTorch Float Tensor, optional): Hidden state matrix for all nodes.

        Return types:
            * **H** (PyTorch Float Tensor): Hidden state matrix for all nodes.
        """
        H_accum = 0
        H_sequence_outputs = []

        probs = torch.nn.functional.softmax(self._attention, dim=0)

        H_previous = H

        for period in range(self.periods):
            X_current = X[:, :, :, period] # shape: [batch_size, num_nodes, feature_dim]. 이런 식으로 반복문 돌리면 period와 같은 차원은 없어짐(축소)
            H_current = self._base_tgcn(X_current, edge_index, edge_weight, H_previous)
            H_previous = H_current
            H_sequence_outputs.append(probs[period] * H_current)

        H_accum = sum(H_sequence_outputs)

        return H_accum

In [11]:
import torch
import torch.nn as nn
import pandas as pd

class EntityEmbedding(torch.nn.Module):
    def __init__(self, col_dims: list, col_list: list):
        '''
        Args:
            cat_dims: 변수별 범주의 개수 리스트
            col_list: 원본 데이터프레임에서 변수들의 순서 왼->오
        '''
        super().__init__()
        self.col_dims = col_dims
        self.col_list = col_list
        # proj_dim must be calculated with the final, corrected col_dims,
        # so we do it here with the provided ones, but it will be re-calculated in the test block
        self.proj_dim = int(max(self.col_dims)**0.5) if self.col_dims else 1

        self.embs = nn.ModuleList([
            nn.Embedding(num_categories, self.proj_dim)
            for num_categories in self.col_dims
        ])

    def forward(self, x_cats):
        '''
        This forward method assumes x_cats is a 2-D tensor of shape [N, F]
        where N is batch_size and F is number of features.
        The current data pipeline produces a different shape, so this is not used.
        '''
        # This logic is for a different data shape and is preserved for potential future use.
        x_cats = x_cats.long()
        outs = []
        for i, emb in enumerate(self.embs):
            out = emb(x_cats[:, i])
            outs.append(out)
        outs_tensor = torch.stack(outs, dim = 1)
        return outs_tensor

class EntityEmbeddingBatch2(EntityEmbedding):
    '''
    for Tensor based process
    '''
    def forward(self, batch: torch.Tensor):
        '''
        Args:
            batch (torch.Tensor): shape: [BATCH_SIZE, num_var(=72)]
        '''
        embedded_list = []
        for idx, emb_func in enumerate(self.embs):
            current_input = batch[:, idx] # shape: [BATCH_SIZE]
            embedded_vec = emb_func(current_input)
            embedded_list.append(embedded_vec)
        return torch.stack(embedded_list, dim=1) # shape: [BATCH_SIZE, NUM_VAR, FEATURE_DIM] (=[32, 72, 25])


In [12]:
import torch
import torch.nn as nn
import sys
import os

def _to_temporal(x_sliced: torch.Tensor, # shape: [num_var, feature_dim] (=[72, 25])
                ad_col_index: list,
                dis_col_index: list,
                los,
                device):

    ad_idx_t = torch.tensor(ad_col_index).to(device)
    dis_idx_t = torch.tensor(dis_col_index).to(device)

    ad_tensor = torch.index_select(x_sliced, dim=0, index=ad_idx_t)
    dis_tensor = torch.index_select(x_sliced, dim=0, index=dis_idx_t)

    tensor_list = [ad_tensor for _ in range(los-1)]
    tensor_list.append(dis_tensor) # tensor_listtensor_listtensor_listtensor_listtensor_listtensor_list 33333333 33
    temp_tensor = torch.stack(tensor_list, dim=-1)

    num_nodes = len(ad_col_index)
    num_features = x_sliced.shape[1] # x_sliced shape: [num_var, feature_dim] (=[72, 25])

    zero = torch.zeros((num_nodes, num_features, 37-los), device=device) # 37: max los

    return torch.concatenate((temp_tensor, zero), dim=-1)

def to_temporal(x_tensor: torch.Tensor, # shape: [batch_size, num_var, feature_dim] (=[32, 72, 25])
                ad_col_index: list,
                dis_col_index: list,
                LOS: torch.Tensor,
                device):
    batch_size = x_tensor.shape[0]

    temp_list = []

    for i in range(batch_size):
        x_sliced = x_tensor[i, :, :]
        los = LOS[i].item()
        # temp_tensor shape: [60, 25, 37]
        temp_tensor = _to_temporal(x_sliced=x_sliced,
                                   ad_col_index=ad_col_index,
                                   dis_col_index=dis_col_index,
                                   los=los,
                                   device=device)
        temp_list.append(temp_tensor)
    return torch.stack(temp_list, dim=0) # shape: [32, 60, 25, 37]



class A3TGCNCat1(nn.Module):
    '''
    tensor 연산 위주로 수행하는 모델
    '''
    def __init__(self, batch_size, col_list, col_dims, hidden_channel):
        '''
        Args:
            col_info(list): [col_dims, col_list]
                            col_list(list): 데이터에서 나타나는 변수의 순서
                            col_dims(list): 각 변수 별 범주의 개수, 순서는 col_list를 따라야 함
            num_layers(int): TGCN 레이어의 개수
            hidden_channel(int): TGCN의 hidden channel
        '''
        super().__init__()
        self.batch_size = batch_size
        self.col_dims = col_dims
        self.col_list = col_list
        self.hidden_channel = hidden_channel

        # EntityEmbedding 레이어 정의
        self.entity_embedding_layer = EntityEmbeddingBatch2(col_dims=self.col_dims,
                                                 col_list=self.col_list)

        # A3TGCN2 레이어 정의
        a3tgcn_input_channel = self.entity_embedding_layer.proj_dim

        self.a3tgcn_layer = A3TGCN2(in_channels=a3tgcn_input_channel,
                        out_channels=hidden_channel,
                        periods=37,
                        batch_size=batch_size)

        # 분류기 레이어 정의
        self.classifier_b = nn.Sequential(
            nn.Linear(hidden_channel, hidden_channel * 2),
            nn.ReLU(),
            nn.Linear(hidden_channel * 2, 2)
        )

    def forward(self, ad_col_index, dis_col_index, x_batch: torch.Tensor, LOS_batch: torch.Tensor, template_edge_index: torch.Tensor, device):
        '''
        Args:
            batch(torch.Tensor): X, y만 정의되어 있는 Data 객체,
            template_edge_index(torch.Tensor): edge_index는 동일하므로 template_edge_index로 한꺼번에 전달
        '''
        x_embedded = self.entity_embedding_layer(x_batch)
        x = to_temporal(x_embedded, ad_col_index, dis_col_index, LOS_batch, device)
        after_GNN = self.a3tgcn_layer(x, template_edge_index) # [32, 60, 32]

        # global mean pooling [32, 60, 32] -> [32, 32]
        mean_pooled = torch.mean(after_GNN, dim=1)

        return self.classifier_b(mean_pooled)


In [13]:
from torch.utils.data import random_split

def train_test_split_customed(dataset, ratio=[0.7, 0.15, 0.15], seed=42, num_workers=8):

    train_dataset, val_dataset, test_dataset = random_split(
        dataset=dataset,
        lengths=ratio,
        generator=torch.Generator().manual_seed(seed)
    )

    print(f"Train Set Size: {len(train_dataset)}")
    print(f"Test Set Size: {len(test_dataset)}")

    train_dataloader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers)
    val_dataloader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)
    test_dataloader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)

    return train_dataloader, val_dataloader, test_dataloader

In [14]:
from tqdm.notebook import tqdm

def train(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for x_batch, y_batch, los_batch in tqdm(dataloader, desc="train_process", leave=True):
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        los_batch = los_batch.to(device)

        optimizer.zero_grad()

        result = model(
            ad_col_index,
            dis_col_index,
            x_batch,
            los_batch,
            edge_index,
            device
        )
        loss = criterion(result, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x_batch.size(0)

        epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss

In [15]:
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from tqdm.notebook import tqdm

def evaluate(model, val_dataloader, criterion, device, ad_col_index, dis_col_index, edge_index):
    model.eval()
    running_loss = 0.0
    total_correct = 0
    total_samples = 0

    # 지표 계산을 위해 모든 레이블과 예측값, 로짓을 저장할 리스트 초기화
    all_targets = []
    all_predictions = []
    all_scores = []

    with torch.no_grad():
        for x_batch, y_batch, los_batch in tqdm(val_dataloader, desc="eval_process", leave=True):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            los_batch = los_batch.to(device)

            # 1. 순전파
            result = model(
                ad_col_index,
                dis_col_index,
                x_batch,
                los_batch,
                edge_index,
                device
            )

            # 2. 손실 계산
            loss = criterion(result, y_batch)
            running_loss += loss.item() * x_batch.size(0)

            # 3. 예측 및 로짓 저장
            # 클래스 인덱스 (Predicted Label)
            _, predicted = torch.max(result, 1)

            # 로짓 또는 확률 (AUC 계산에 사용)
            # CrossEntropyLoss의 출력은 로짓(Logits)입니다.
            # 이진 분류 (2개 클래스)라면 Softmax 후 클래스 1의 확률을 사용하거나,
            # 다중 분류라면 로짓 자체를 roc_auc_score에 전달할 수 있습니다.
            # 여기서는 로짓(result)을 그대로 사용하고, AUC 계산 시 멀티클래스 지원 옵션을 적용합니다.

            all_targets.append(y_batch.cpu().numpy())
            all_predictions.append(predicted.cpu().numpy())
            all_scores.append(result.cpu().numpy())

            # 4. 정확도 누적
            total_correct += (predicted == y_batch).sum().item()
            total_samples += y_batch.size(0)

    # 모든 배치 데이터 합치기
    all_targets = np.concatenate(all_targets)
    all_predictions = np.concatenate(all_predictions)
    all_scores = np.concatenate(all_scores)

    # 5. 최종 지표 계산
    epoch_loss = running_loss / len(val_dataloader.dataset)
    epoch_accuracy = total_correct / total_samples

    # 정밀도, 재현율, F1-Score 계산
    # average='macro'는 각 클래스별 점수를 계산한 후 평균을 내는 방식입니다.
    # zero_division=0은 나눗셈 오류 방지 (해당 클래스에 샘플이 없을 경우 0으로 처리)
    epoch_precision = precision_score(all_targets, all_predictions, average='macro', zero_division=0)
    epoch_recall = recall_score(all_targets, all_predictions, average='macro', zero_division=0)
    epoch_f1 = f1_score(all_targets, all_predictions, average='macro', zero_division=0)

    # AUC 계산
    # multi_class='ovr' (One-vs-Rest) 방식은 다중 분류에서 가장 흔하게 사용됩니다.
    try:
        epoch_auc = roc_auc_score(all_targets, all_scores, multi_class='ovr')
    except ValueError:
        # 단일 클래스만 존재하거나 (AUC 계산 불가능) 이진 분류가 아닌 경우를 대비
        print("Warning: AUC score could not be calculated (requires multiple classes or specific binary format).")
        epoch_auc = 0.0

    # 6. 모든 지표 반환
    return epoch_loss, epoch_accuracy, epoch_precision, epoch_recall, epoch_f1, epoch_auc

In [16]:
def save_checkpoint(epoch, model, optimizer, scheduler, best_loss, filename):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_loss': best_loss,
    }
    torch.save(checkpoint, filename)

In [19]:
EPOCH = 100
scheduler_patience = 10
early_stopping_patience = 15
best_val_loss = float('inf')
model_path = os.path.join(root, '/model/best_model.pt')
device = device_set()
BATCH_SIZE = 128
num_workers=16
root = './data_tensor_cache'


from torch.optim.lr_scheduler import ReduceLROnPlateau

dataset = TEDSTensorDataset(root)

train_dataloader, val_dataloader, test_dataloader = train_test_split_customed(dataset, num_workers=num_workers)

col_list, col_dims, ad_col_index, dis_col_index = dataset.col_info

model = A3TGCNCat1(batch_size=BATCH_SIZE, col_list=col_list,
                    col_dims=col_dims, hidden_channel=64)
model.to(device)

mi_dict_path = os.path.join(root, 'data', 'mi_dict_static.pickle')
edge_index = mi_edge_index_batched(batch_size=BATCH_SIZE,
                                        mi_dict_path=mi_dict_path,
                                        top_k=6,
                                        return_edge_attr=False)
edge_index = edge_index.to(device)

counter = 0

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = ReduceLROnPlateau(optimizer, "min", patience=10)


Using device: mps
저장되어 있는 전처리된 데이터가 있습니다. 해당 데이터를 불러오는 중..
불러오기 완료
Train Set Size: 975897
Test Set Size: 209120
Using device: mps


/opt/miniconda3/envs/pyg_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 14 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


FileNotFoundError: [Errno 2] No such file or directory: './data_tensor_cache/data/mi_dict_static.pickle'

In [20]:
from tqdm import tqdm
import torch

for epoch in tqdm(range(EPOCH)):
    # 1. 훈련
    train_loss = train(model, train_dataloader, criterion, optimizer, device)

    # 2. 평가
    result = evaluate(model, val_dataloader, criterion, device, ad_col_index, dis_col_index, edge_index)
    val_loss, val_accuracy, val_precision, val_recall, val_f1, val_auc = result

    # 3. 결과 출력
    print(f"Epoch {epoch+1}/{EPOCH}, Train Loss: {train_loss:.4f}")
    print(f"Epoch {epoch+1}/{EPOCH}, Val Loss: {val_loss:.4f}, Acc: {val_accuracy:.4f}, F1: {val_f1:.4f}, AUC: {val_auc:.4f}")

    # 4. 스케줄러 업데이트
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        print("Validation Loss 개선. 체크포인트 저장.")
        best_val_loss = val_loss
        save_checkpoint(epoch + 1, model, optimizer, scheduler, best_val_loss, model_path)
        counter = 0 # 카운터 초기화
    else:
        counter += 1
        print(f"Validation Loss 개선 없음. Patience: {counter}/{early_stopping_patience}")

        if counter >= early_stopping_patience:
            print(f"조기 종료! {early_stopping_patience} 에폭 동안 성능 개선 없음.")
            break

  0%|          | 0/100 [00:00<?, ?it/s]


NameError: name 'criterion' is not defined

In [ ]:
import torch
print(f"사용 가능한 GPU 개수: {torch.cuda.device_count()}개")